# **BGM CGM Alignment**

Compares fingerstick (BGM) readings with the nearest continuous glucose monitor (CGM) values within a 15-minute window and calculates the percent difference to measure alignment. Uses pre-processed BGM and CGM datasets and exports the results to CSV for Tableau visualization.

Run on Python 3.15 | No errors | No warnings

In [1]:
# Import packages

# For data manipulation
lazy import numpy as np
lazy import pandas as pd

# For working with files
lazy import glob
lazy import os

# For working with datetime objects
lazy from datetime import datetime

In [2]:
# Find the BGM dataset

# Define the directory and filename pattern
directory = r"D:\OneDrive\Documents\Python\Current\Blood Glucose Readings\BGM"
pattern = "Blood Glucose Readings for Analysis.csv"

# Construct the full search pattern
search_pattern = os.path.join(directory, pattern)

# Find all matching files
matching_files = glob.glob(search_pattern)

# Ensure there is at least one match
if not matching_files:
    raise SystemExit("No matching files found. Exiting program.")

# Sort files by modification time to get the most recent one
latest_file = max(matching_files, key=os.path.getmtime)

# Extract just the file name
file_name = os.path.basename(latest_file)

print(f"Found file: {file_name}")

Found file: Blood Glucose Readings for Analysis.csv


In [3]:
# Load BGM dataset
df0 = pd.read_csv(latest_file, usecols=['Date', 'Time', 'DateTime', 'Value'])

In [4]:
# Find the CGM dataset

# Define the directory and filename pattern
directory = r"D:\OneDrive\Documents\Python\Current\Blood Glucose Readings\CGM"
pattern = "Clarity Readings for Analysis.csv"

# Construct the full search pattern
search_pattern = os.path.join(directory, pattern)

# Find all matching files
matching_files = glob.glob(search_pattern)

# Ensure there is at least one match
if not matching_files:
    raise SystemExit("No matching files found. Exiting program.")

# Sort files by modification time to get the most recent one
latest_file = max(matching_files, key=os.path.getmtime)

# Extract just the file name
file_name = os.path.basename(latest_file)

print(f"Found file: {file_name}")

Found file: Clarity Readings for Analysis.csv


In [5]:
# Load CGM dataset
df1 = pd.read_csv(latest_file, usecols=['Date', 'Time', 'DateTime', 'Value'])

In [6]:
# Align BGM readings with CGM readings within a 15-minute time window

# Convert the 'DateTime' columns to datetime format
df0['DateTime'] = pd.to_datetime(df0['DateTime'])
df1['DateTime'] = pd.to_datetime(df1['DateTime'])

# Define a function to find CGM readings within 15 minutes before and after each BGM reading
def find_cgm_within_time_window(bgm_time, df1, time_window_minutes=15):
    # Define the time window
    start_time = bgm_time - pd.Timedelta(minutes=time_window_minutes)
    end_time = bgm_time + pd.Timedelta(minutes=time_window_minutes)
    
    # Filter CGM readings that fall within the time window
    filtered_cgm = df1[(df1['DateTime'] >= start_time) & (df1['DateTime'] <= end_time)]
    return filtered_cgm

# Apply this function to each BGM reading and store the results
bgm_cgm_matches = []

for _, row in df0.iterrows():
    matching_cgm = find_cgm_within_time_window(row['DateTime'], df1)
    if not matching_cgm.empty:
        # For each BGM reading, store the matched CGM readings
        for _, cgm_row in matching_cgm.iterrows():
            bgm_cgm_matches.append({
                'BGM DateTime': row['DateTime'],
                'BGM Value': row['Value'],
                'CGM DateTime': cgm_row['DateTime'],
                'CGM Value': cgm_row['Value']
            })

# Convert the results into a DataFrame
df2 = pd.DataFrame(bgm_cgm_matches)

In [7]:
# Display the first 5 rows of the combined DataFrame
df2.head()

,BGM DateTime,BGM Value,CGM DateTime,CGM Value
0,2026-08-26 06:13:00,104,2026-08-26 05:59:13,115
1,2026-08-26 06:13:00,104,2026-08-26 06:04:13,108
2,2026-08-26 06:13:00,104,2026-08-26 06:09:14,107
3,2026-08-26 06:13:00,104,2026-08-26 06:14:13,112
4,2026-08-26 06:13:00,104,2026-08-26 06:19:13,116


In [8]:
# Display basic information about the combined DataFrame
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BGM DateTime  1080 non-null   datetime64[us]
 1   BGM Value     1080 non-null   int64         
 2   CGM DateTime  1080 non-null   datetime64[us]
 3   CGM Value     1080 non-null   int64         
dtypes: datetime64[us](2), int64(2)
memory usage: 33.9 KB


In [9]:
# Remove the seconds from the DateTime columns
df2['BGM DateTime'] = df2['BGM DateTime'].dt.floor('min')
df2['CGM DateTime'] = df2['CGM DateTime'].dt.floor('min')

In [10]:
# Delete rows prior to 9/1/2024 (CGM data start date)
df2 = df2[df2['BGM DateTime'] > '2024-08-30']

In [11]:
# Add a Date column to df2 that is the date of the BGM DateTime
df2['Date'] = df2['BGM DateTime'].dt.date

In [12]:
# Add a column for difference
df2['Difference'] = df2['CGM Value'] - df2['BGM Value']

In [13]:
# Add a column for the absolute value of the difference
df2['Abs Difference'] = df2['Difference'].abs()

In [14]:
# Verify the changes
df2.head()

,BGM DateTime,BGM Value,CGM DateTime,CGM Value,Date,Difference,Abs Difference
0,2026-08-26 06:13:00,104,2026-08-26 05:59:00,115,2026-08-26,11,11
1,2026-08-26 06:13:00,104,2026-08-26 06:04:00,108,2026-08-26,4,4
2,2026-08-26 06:13:00,104,2026-08-26 06:09:00,107,2026-08-26,3,3
3,2026-08-26 06:13:00,104,2026-08-26 06:14:00,112,2026-08-26,8,8
4,2026-08-26 06:13:00,104,2026-08-26 06:19:00,116,2026-08-26,12,12


In [15]:
# Find the minimum Abs Difference for each BGM DateTime
df3 = df2.loc[df2.groupby('BGM DateTime')['Abs Difference'].idxmin()]

In [16]:
# Range is Difference / BGM Value
df3['Range'] = df3['Difference'] / df3['BGM Value']
# Round the Range to 2 decimal places
df3['Range'] = df3['Range'].round(2)

In [17]:
# Calculate the Range Within using a min value of 0.10
range_abs = np.maximum(np.ceil(np.abs(df3['Range']) * 10) / 10, 0.10)
df3['Range Within'] = np.where(df3['Range'] == 0, 0.10, np.sign(df3['Range']) * range_abs)

In [18]:
# Load location rules
df4 = pd.read_excel('CGM Locations.xlsx', sheet_name='Locations')

In [19]:
# Convert 'Date' column to datetime
df4['Date'] = pd.to_datetime(df4['Date'])

In [20]:
# Convert Date and Time columns in df4 to a single datetime column
df4['DateTime'] = pd.to_datetime(df4['Date'].astype(str) + " " + df4['Time'].astype(str))

In [21]:
# Sort both DataFrames by DateTime
df4 = df4.sort_values('DateTime')
df3 = df3.sort_values('BGM DateTime')

In [22]:
# Perform an asof merge to assign the most recent CGM location
df5 = pd.merge_asof(
    df3,
    df4[['DateTime', 'Location']],
    left_on='BGM DateTime',
    right_on='DateTime',
    direction='backward'
)

In [23]:
# Show the last few rows of df5 to inspect the most recent data
df5.tail(10)

,BGM DateTime,BGM Value,CGM DateTime,CGM Value,Date,Difference,Abs Difference,Range,Range Within,DateTime,Location
170,2026-11-19 06:17:00,86,2026-11-19 06:07:00,96,2026-11-19,10,10,0.12,0.2,2026-11-12 13:27:00,Right - Middle Third
171,2026-11-19 19:02:00,120,2026-11-19 19:07:00,130,2026-11-19,10,10,0.08,0.1,2026-11-12 13:27:00,Right - Middle Third
172,2026-11-20 06:16:00,94,2026-11-20 06:07:00,97,2026-11-20,3,3,0.03,0.1,2026-11-12 13:27:00,Right - Middle Third
173,2026-11-20 19:03:00,129,2026-11-20 18:52:00,130,2026-11-20,1,1,0.01,0.1,2026-11-12 13:27:00,Right - Middle Third
174,2026-11-21 07:14:00,96,2026-11-21 07:07:00,96,2026-11-21,0,0,0.00,0.1,2026-11-12 13:27:00,Right - Middle Third
175,2026-11-21 19:06:00,92,2026-11-21 19:07:00,95,2026-11-21,3,3,0.03,0.1,2026-11-12 13:27:00,Right - Middle Third
176,2026-11-22 08:36:00,93,2026-11-22 08:47:00,106,2026-11-22,13,13,0.14,0.2,2026-11-12 13:27:00,Right - Middle Third
177,2026-11-22 19:01:00,88,2026-11-22 19:02:00,109,2026-11-22,21,21,0.24,0.3,2026-11-12 13:27:00,Right - Middle Third
178,2026-11-23 06:12:00,83,2026-11-23 06:07:00,97,2026-11-23,14,14,0.17,0.2,2026-11-12 13:27:00,Right - Middle Third
179,2026-11-23 19:05:00,104,2026-11-23 19:12:00,111,2026-11-23,7,7,0.07,0.1,2026-11-12 13:27:00,Right - Middle Third


In [24]:
# Create a column for AM/PM based on the BGM DateTime
df5['AM/PM'] = df5['BGM DateTime'].dt.strftime('%p')    

In [25]:
# Order the columns in df5
df5 = df5[['Date', 'AM/PM', 'BGM DateTime', 'BGM Value', 'CGM DateTime', 'CGM Value', 'Difference', 'Abs Difference', 'Range', 'Range Within', 'Location']]

In [26]:
# Replace NaN values in 'Location' with 'Unknown'
df5['Location'] = df5['Location'].fillna('Unknown')

In [27]:
# Set date to datetime format
df5['Date'] = pd.to_datetime(df5['Date'])

In [28]:
# Creat a column for device
conditions = [
    (df5['Date'] > '2026-01-23')
]

choices = [
    'True Metrix Air'
]

df5['Device'] = np.select(conditions, choices, default='OneTouch Verio Flex')

In [29]:
# Display the first 5 rows of df5
df5.head()

,Date,AM/PM,BGM DateTime,BGM Value,CGM DateTime,CGM Value,Difference,Abs Difference,Range,Range Within,Location,Device
0,2026-08-26,AM,2026-08-26 06:13:00,104,2026-08-26 06:09:00,107,3,3,0.03,0.1,Right - Middle Third,True Metrix Air
1,2026-08-26,PM,2026-08-26 19:00:00,131,2026-08-26 19:09:00,131,0,0,0.00,0.1,Right - Middle Third,True Metrix Air
2,2026-08-27,AM,2026-08-27 06:13:00,102,2026-08-27 06:09:00,115,13,13,0.13,0.2,Right - Middle Third,True Metrix Air
3,2026-08-27,PM,2026-08-27 19:05:00,144,2026-08-27 18:54:00,148,4,4,0.03,0.1,Right - Middle Third,True Metrix Air
4,2026-08-28,AM,2026-08-28 06:13:00,101,2026-08-28 06:04:00,106,5,5,0.05,0.1,Right - Middle Third,True Metrix Air


In [30]:
# Display basic information about df5
df5.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            180 non-null    datetime64[s] 
 1   AM/PM           180 non-null    str           
 2   BGM DateTime    180 non-null    datetime64[us]
 3   BGM Value       180 non-null    int64         
 4   CGM DateTime    180 non-null    datetime64[us]
 5   CGM Value       180 non-null    int64         
 6   Difference      180 non-null    int64         
 7   Abs Difference  180 non-null    int64         
 8   Range           180 non-null    float64       
 9   Range Within    180 non-null    float64       
 10  Location        180 non-null    str           
 11  Device          180 non-null    str           
dtypes: datetime64[s](1), datetime64[us](2), float64(2), int64(4), str(3)
memory usage: 17.0 KB


In [31]:
# Save df5 to CSV
df5.to_csv('BGM CGM Range.csv', index=False)